# Ottawa Company Analytics-Role Categorization (clean)

**Categorize** (rate-limited, cached): use Gemini 3.1 Flash-Lite **with Google Search grounding** to profile each company from real web data, not the name alone.

Design notes:
- Writes go to a separate **Results** worksheet and are **checkpointed** — a dropped Colab session resumes instead of restarting. The source sheet is never cleared.
- Grounding is **opt-in** via `USE_GROUNDING`. The model only touches the web when the `google_search` tool is attached.


In [ ]:
# --- Install (new google-genai SDK; legacy google-generativeai is a dead end for 3.x) ---
!pip install -q --upgrade google-genai gspread


In [ ]:
# --- Imports ---
import re, time, json, threading

import gspread
from google.colab import auth, userdata
from google.auth import default as gauth_default

from google import genai
from google.genai import types

In [ ]:
# --- CONFIG ---
SOURCE_SHEET    = "Company List"      # cols: A=Company Name, B=Website
RESULTS_SHEET   = "Results"       # created if missing; never cleared destructively
AUDIT_SHEET     = "ATS_Audit"     # Stage-1 output (ats_audit.py): career_url + ATS

MODEL_ID        = "gemini-3.1-flash-lite"   # GA id; the -preview variant was shut down 2026-05-25
USE_GROUNDING   = True            # attach google_search tool so the model uses the live web

CHECKPOINT_EVERY = 15             # flush to the sheet every N processed companies

LLM_SLEEP       = 2.0             # base pause between grounded calls
LLM_MAX_RETRIES = 4

# Results-worksheet schema. Stage-1 discovery fills cols 0-5; this notebook writes 6-11.
RESULTS_HEADERS = ["Company Name", "Website", "Career Page URL", "Detected ATS",
                   "Board URL", "Audit Status", "Category", "Analytics Context",
                   "Estimated Size", "Analytics Roles Likely", "Evidence", "Grounded Queries"]


In [ ]:
# --- Auth: Google Sheets + Gemini ---
auth.authenticate_user()
creds, _ = gauth_default()
gc = gspread.authorize(creds)
SPREADSHEET_URL = userdata.get("SPREADSHEET_URL")
assert SPREADSHEET_URL, "Add the source spreadsheet link to Colab Secrets as 'SPREADSHEET_URL'."
sh = gc.open_by_url(SPREADSHEET_URL)
src_ws = sh.worksheet(SOURCE_SHEET)
print("Source sheet:", SOURCE_SHEET, "| rows:", src_ws.row_count)

client = genai.Client(vertexai=True, project="job-search-pipeline-prod", location="global")
print("Gemini client ready. Grounding:", USE_GROUNDING)


In [ ]:
def normalize_domain(raw: str) -> str:
    d = (raw or "").strip().lower()
    if not d:
        return ""
    if "://" in d:
        d = d.split("://", 1)[1]
    return d.split("/")[0]

# --- Stage handoff: read Stage-1 discovery (career page + ATS) from the ATS_Audit tab ---
# ats_audit.py emits 7 cols: Company Name, Website, Career Page URL, Detected ATS,
# Board Token, Found Via, Status. Import ats_audit_results.csv into AUDIT_SHEET first.
def get_or_create_results_ws():
    try:
        ws = sh.worksheet(RESULTS_SHEET)
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=RESULTS_SHEET, rows=1000, cols=len(RESULTS_HEADERS))
        ws.update(range_name="A1", values=[RESULTS_HEADERS])
    return ws

audit_ws = sh.worksheet(AUDIT_SHEET)
audit_rows = [r for r in audit_ws.get_all_values()[1:] if r and r[0].strip()]
res_ws = get_or_create_results_ws()

# Resume: keep prior categorization (Category..Grounded Queries) keyed by website.
existing = res_ws.get_all_values()
done = {}
if len(existing) > 1:
    for row in existing[1:]:
        row = (row + [""] * len(RESULTS_HEADERS))[:len(RESULTS_HEADERS)]
        key = normalize_domain(row[1])
        if key and row[6]:               # already categorized
            done[key] = row

# records = 12-col RESULTS layout. Cols 0-5 come from ATS_Audit; 6-11 filled by this notebook.
records = []
for a in audit_rows:
    a = (a + [""] * 7)[:7]
    name, website, career_url, ats, token, via, status = a
    key = normalize_domain(website)
    if key in done:
        records.append(done[key])
    else:
        records.append([name, website, career_url, ats, token, status,
                        "", "", "", "", "", ""])

need = sum(1 for r in records if r[2] and not r[6])   # has careers page, not yet categorized
print(f"{len(records)} companies from {AUDIT_SHEET} | {len(done)} already categorized | "
      f"{need} to categorize")


In [ ]:
# --- Safe write: overwrite the Results range in place (never clear the source) ---
_write_lock = threading.Lock()

def flush(records):
    with _write_lock:
        res_ws.update(range_name="A1", values=[RESULTS_HEADERS] + records)


In [ ]:
# --- Grounded categorizer: static instructions live in the system prompt;
# each call passes only the per-company fields. Real web data via google_search. ---
SYSTEM_INSTRUCTION = """Assess whether a company would PLAUSIBLY EVER employ data/analytics staff (BI, analytics engineering, data engineering, data science, product analytics) as an ongoing STRUCTURAL trait of the business — NOT whether it has an opening right now.

Use Google Search to find CURRENT, REAL information about the specific company (what it does, size, data/tech maturity). Do not judge from the name alone.

Rules for "analytics_roles_likely":
- "Yes"   = clearly runs data/analytics functions (software/SaaS, fintech, health-tech, data-heavy product, or mid+ size with real data operations).
- "Maybe" = plausible but unconfirmed, OR you could not find enough to decide. Default to "Maybe" when evidence is thin. NEVER guess "No".
- "No"    = structurally unlikely to EVER need analysts (small local trades, retail, hospitality, single-location services with no data operation).

Return ONLY a JSON object, no markdown, with these keys:
  "category": short industry / software category
  "analytics_context": <=20 words on their data types / needs
  "estimated_size": one of "1-10","11-50","51-200","201-500","501-1000","1000+"
  "analytics_roles_likely": "Yes" | "No" | "Maybe"
  "evidence": <=15 words on what you actually found; write "name-only" if you could not verify anything

Examples:
Fullscript - health-tech platform, 500+ - Yes
Gadget.dev - developer tooling, ~30 people - Maybe
3 Brothers Shawarma - local restaurant - No"""


def _extract_json(text: str):
    text = text.replace("```json", "").replace("```", "").strip()
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None


def _grounding_queries(resp):
    try:
        gm = resp.candidates[0].grounding_metadata
        return list(gm.web_search_queries or [])
    except Exception:
        return []


def categorize_company(name, website, ats, career_url):
    # Only the variable fields go in each call; the rules live in SYSTEM_INSTRUCTION.
    contents = (
        f"Company: {name}\n"
        f"Website: {website}\n"
        f"Careers page: {career_url or 'unknown'}\n"
        f"Detected ATS: {ats}"
    )

    tools = [types.Tool(google_search=types.GoogleSearch())] if USE_GROUNDING else None
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        tools=tools,
        temperature=1.0 if USE_GROUNDING else 0.2,
    )

    for attempt in range(1, LLM_MAX_RETRIES + 1):
        try:
            resp = client.models.generate_content(model=MODEL_ID, contents=contents, config=config)
            data = _extract_json(resp.text or "")
            if data:
                q = _grounding_queries(resp)
                return {
                    "category": str(data.get("category", "")).strip(),
                    "context":  str(data.get("analytics_context", "")).strip(),
                    "size":     str(data.get("estimated_size", "")).strip(),
                    "roles":    str(data.get("analytics_roles_likely", "")).strip(),
                    "evidence": str(data.get("evidence", "")).strip(),
                    "queries":  "; ".join(q) if q else ("(none)" if USE_GROUNDING else "off"),
                }
            print(f"  parse miss for {name}, retry {attempt}")
        except Exception as e:
            msg = str(e)
            if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                wait = LLM_SLEEP * (2 ** attempt)
                print(f"  rate limited on {name}; sleeping {wait:.0f}s")
                time.sleep(wait)
                continue
            print(f"  error on {name}: {type(e).__name__}: {msg[:120]}")
            break
    return {"category": "N/A", "context": "N/A", "size": "N/A", "roles": "N/A",
            "evidence": "categorize failed", "queries": ""}


In [ ]:
# --- Categorize companies that have a career page and no category yet ---
to_cat = [i for i, r in enumerate(records) if r[2] and (not r[6] or r[6] == "N/A")] # has career_url, Category empty
print(f"Categorizing {len(to_cat)} companies (grounding={USE_GROUNDING})...")

for n, i in enumerate(to_cat, 1):
    name, website, ats = records[i][0], records[i][1], records[i][3]
    c = categorize_company(name, website, ats, records[i][2])
    records[i][6]  = c["category"]
    records[i][7]  = c["context"]
    records[i][8]  = c["size"]
    records[i][9]  = c["roles"]
    records[i][10] = c["evidence"]
    records[i][11] = c["queries"]
    print(f"  [{n}/{len(to_cat)}] {name}: {c['roles']} | {c['size']} | queries={c['queries'][:60]}")
    if n % CHECKPOINT_EVERY == 0:
        flush(records)
        print("    ...checkpointed")
    time.sleep(LLM_SLEEP)

flush(records)
print("Categorization complete.")


In [ ]:
# --- Summary ---
from collections import Counter
grounded    = sum(1 for r in records if r[11] and r[11] not in ("", "off", "(none)"))
print(f"Categorized w/ web:   {grounded}")
print()
print("Roles likely:", dict(Counter(r[9] for r in records if r[9])))
